In [11]:
# Load the data
import pandas as pd

df = pd.read_csv("../data/creditcard.csv")
print(df.shape)

(284807, 31)


In [13]:
# Check data quality
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 0
Duplicate rows: 1081


In [14]:
# How rare is fraud?
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True) * 100)

Class
0    284315
1       492
Name: count, dtype: int64
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


In [15]:
# Look at Amount for each class
print(df.groupby("Class")["Amount"].describe())

'''This shows summary numbers for Amount, split by normal and fraud.
You should see: amounts are mostly small, a few are huge (max 25,691), and fraud amounts overlap with normal ones. So Amount alone can’t catch fraud.'''

          count        mean         std  min   25%    50%     75%       max
Class                                                                      
0      284315.0   88.291022  250.105092  0.0  5.65  22.00   77.05  25691.16
1         492.0  122.211321  256.683288  0.0  1.00   9.25  105.89   2125.87


In [16]:
# Remove Duplicates
df = df.drop_duplicates()
print(df.shape)

(283726, 31)


In [18]:
# Split into practice set and exam set
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class", "Time"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print("Frauds in test set:", y_test.sum())

(226980, 29) (56746, 29)
Frauds in test set: 95


In [19]:
# Scale Amount
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train["Amount"] = scaler.fit_transform(X_train[["Amount"]])
X_test["Amount"] = scaler.transform(X_test[["Amount"]])

print(X_train["Amount"].mean().round(3))

0.0


In [23]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

pred = baseline.predict(X_train)

print("Accuracy:", round(accuracy_score(y_train, pred), 4))
print("Fraud caught (recall):", recall_score(y_train, pred))

"""
1. DummyClassifier(strategy="most_frequent") creates a model that always predicts whichever class is most common in the data. Here that’s “not fraud”.
2. baseline.fit(...) lets it “learn”. All it learns is which class is most common.
3. baseline.predict(...) gets its answers for every row.
4. accuracy_score is the share of answers that were correct.
5. recall_score is the share of real fraud cases the model caught.

A model that always predicts not fraud gets 99.8% accuracy but catches 0% of fraud, so accuracy is misleading on imbalanced data.

"""

Accuracy: 0.9983
Fraud caught (recall): 0.0


'\n1. DummyClassifier(strategy="most_frequent") creates a model that always predicts whichever class is most common in the data. Here that’s “not fraud”.\n2. baseline.fit(...) lets it “learn”. All it learns is which class is most common.\n3. baseline.predict(...) gets its answers for every row.\n4. accuracy_score is the share of answers that were correct.\n5. recall_score is the share of real fraud cases the model caught.\n\nA model that always predicts not fraud gets 99.8% accuracy but catches 0% of fraud, so accuracy is misleading on imbalanced data.\n\n'

In [25]:
# Try Logistic Regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=42
)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_tr, y_tr)

pred = lr.predict(X_val)

print(confusion_matrix(y_val, pred))
print("Precision:", round(precision_score(y_val, pred), 3))
print("Recall:", round(recall_score(y_val, pred), 3))

"""
1. Top-right (false alarms): good transactions we wrongly flagged. These are annoyed customers.(7)
2. Bottom-left (missed fraud): real fraud we let through. These are lost money. (31)
3. Bottom-right: fraud we correctly caught. (63)
"""

[[56644     7]
 [   31    63]]
Precision: 0.9
Recall: 0.67


In [26]:
# Try Decision Tree
from sklearn.tree import DecisionTreeClassifier

for name, depth in [("Unlimited depth", None), ("Max depth 5", 5)]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_tr, y_tr)

    print(name)
    print("  Recall on training set:  ", round(recall_score(y_tr, tree.predict(X_tr)), 3))
    print("  Recall on validation set:", round(recall_score(y_val, tree.predict(X_val)), 3))
    print("  Precision on validation: ", round(precision_score(y_val, tree.predict(X_val)), 3))

Unlimited depth
  Recall on training set:   1.0
  Recall on validation set: 0.798
  Precision on validation:  0.735
Max depth 5
  Recall on training set:   0.831
  Recall on validation set: 0.83
  Precision on validation:  0.886


In [27]:
# Try Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)
rf.fit(X_tr, y_tr)

print("Recall on training set:  ", round(recall_score(y_tr, rf.predict(X_tr)), 3))
print("Recall on validation set:", round(recall_score(y_val, rf.predict(X_val)), 3))
print("Precision on validation: ", round(precision_score(y_val, rf.predict(X_val)), 3))

Recall on training set:   0.838
Recall on validation set: 0.83
Precision on validation:  0.94


In [29]:
# The threshold
probs = rf.predict_proba(X_val)[:, 1]

print("Threshold | Flagged | Precision | Recall")
for t in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    pred_t = (probs >= t).astype(int)
    print(f"{t:>9} | {pred_t.sum():>7} | "
          f"{precision_score(y_val, pred_t):>9.3f} | "
          f"{recall_score(y_val, pred_t):>6.3f}")

Threshold | Flagged | Precision | Recall
      0.1 |     105 |     0.790 |  0.883
      0.2 |      93 |     0.882 |  0.872
      0.3 |      89 |     0.899 |  0.851
      0.5 |      83 |     0.940 |  0.830
      0.7 |      74 |     0.959 |  0.755
      0.9 |      42 |     1.000 |  0.447


In [30]:
from sklearn.metrics import average_precision_score

rf_final = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)
rf_final.fit(X_train, y_train)

test_probs = rf_final.predict_proba(X_test)[:, 1]
test_pred = (test_probs >= 0.3).astype(int)

print(confusion_matrix(y_test, test_pred))
print("Precision:", round(precision_score(y_test, test_pred), 3))
print("Recall:", round(recall_score(y_test, test_pred), 3))
print("PR-AUC:", round(average_precision_score(y_test, test_probs), 3))

[[56644     7]
 [   24    71]]
Precision: 0.91
Recall: 0.747
PR-AUC: 0.786
